# Ant colony optimization — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def sphere(x, target=None):
    x=np.asarray(x, dtype=float)
    if target is None: target=np.zeros_like(x)
    return float(np.sum((x-np.asarray(target,dtype=float))**2))
def rastrigin(x):
    x=np.asarray(x,dtype=float); return float(10*len(x)+np.sum(x*x-10*np.cos(2*np.pi*x)))
def softmax_min(vals, temp=1.0):
    vals=np.asarray(vals,dtype=float); w=np.exp(-(vals-vals.min())/temp); return w/w.sum()
print('setup ok')

## Probabilistic construction from pheromone and heuristic desirability, reinforced by good tours
Ant colony optimization builds solutions one step at a time. Edges become attractive when they have pheromone from previous good solutions and when the heuristic says they are cheap now; evaporation prevents early accidents from ruling forever.

In [ ]:
# 1) Pheromone times heuristic gives transition probabilities
pher=np.ones((4,4)); eta=np.array([[0,1,1/4,1/9],[1,0,1,1/4],[1/4,1,0,1],[1/9,1/4,1,0]],float); avail=[1,2,3]
score=np.array([pher[0,j]*eta[0,j] for j in avail]); prob=score/score.sum()
plt.figure(figsize=(6,3)); plt.bar([f'0->{j}' for j in avail], prob); plt.ylabel('probability'); plt.title('nearer edges are favored before learning')
assert np.allclose(prob, [0.7346938775510203,0.18367346938775508,0.08163265306122448])

In [ ]:
# 2) Alpha controls how strongly pheromone biases choices
base=np.array([1.,1.]); learned=np.array([3.,1.]); alphas=np.array([0.,1.,2.]); probs=[]
for alpha in alphas:
    s=learned**alpha; probs.append(s[0]/s.sum())
plt.figure(figsize=(6,3)); plt.plot(alphas, probs, '-o'); plt.xlabel('pheromone exponent alpha'); plt.ylabel('P(edge A)'); plt.title('higher alpha amplifies accumulated pheromone')
assert probs[0]==0.5 and probs[-1] > probs[1] > probs[0]

In [ ]:
# 3) Evaporation plus deposit updates pheromone
old=np.array([1.,1.,1.]); rho=0.2; deposit=np.array([0.5,0.,1.0]); new=(1-rho)*old+deposit
plt.figure(figsize=(6,3)); plt.bar(['edge 1','edge 2','edge 3'], new); plt.ylabel('new pheromone'); plt.title('unused edges fade; good-tour edges grow')
assert np.allclose(new,[1.3,0.8,1.8])

In [ ]:
# 4) Repeated ants reinforce the shortest tiny tour
rng=np.random.default_rng(11); dist=np.array([[0,1,3,4],[1,0,2,5],[3,2,0,1],[4,5,1,0]],float); pher=np.ones_like(dist); np.fill_diagonal(pher,0); best=[]
for _ in range(25):
    tours=[]
    for _ant in range(12):
        tour=[0]; remaining=[1,2,3]
        while remaining:
            cur=tour[-1]; score=np.array([pher[cur,j]*(1/dist[cur,j])**2 for j in remaining]); p=score/score.sum(); nxt=remaining[rng.choice(len(remaining),p=p)]; tour.append(nxt); remaining.remove(nxt)
        length=sum(dist[tour[i],tour[(i+1)%4]] for i in range(4)); tours.append((length,tour))
    pher*=0.7
    for length,tour in tours:
        for i in range(4): pher[tour[i],tour[(i+1)%4]] += 1/length
    best.append(min(length for length,_ in tours))
plt.figure(figsize=(6,3)); plt.plot(best,'-o'); plt.xlabel('iteration'); plt.ylabel('best tour length'); plt.title('pheromone reinforces good constructed tours')
assert min(best) <= 8.0 and best[-1] <= best[0]

In [ ]:
# 5) Final pheromone network highlights preferred edges
plt.figure(figsize=(4,4)); coords=np.array([[0,0],[1,0],[1,1],[0,1]],float)
for i in range(4):
    for j in range(4):
        if i!=j: plt.plot([coords[i,0],coords[j,0]],[coords[i,1],coords[j,1]],'k-',alpha=min(1,pher[i,j]/pher.max()),lw=1+4*pher[i,j]/pher.max())
plt.scatter(coords[:,0],coords[:,1],s=100,c='tab:blue'); [plt.text(coords[i,0]+.03,coords[i,1]+.03,str(i)) for i in range(4)]; plt.axis('equal'); plt.axis('off'); plt.title('pheromone is a learned routing memory')
assert pher.max() > pher[pher>0].min()